# GPT and Autoregressive Models

## Start with the problem: predict the next token from the prefix

An autoregressive language model factors a sequence into next-token decisions. At each position it uses only the prefix, predicts a distribution over the next token, and repeats. GPT-style models use decoder-only Transformer blocks with causal self-attention.

This lesson derives the training objective, distinguishes it from generation, and explains decoding controls and evaluation limits.

## See the causal generation loop

```text
prefix -> causal Transformer -> next-token distribution
  ^                                  |
  +--------- append selected token <-+
```

For token sequence `[A, B, C]`, a shifted training example uses inputs `[BOS, A, B]` and targets `[A, B, C]`. A causal mask ensures the prediction at each position cannot inspect its target or later tokens.

## Train shifted next-token predictions

For tokens $x_1,\ldots,x_T$, the autoregressive factorization is

$$p(x_1,\ldots,x_T)=\prod_{t=1}^{T}p(x_t\mid x_{<t}).$$

Training minimizes the sum or mean negative log probability of each target token given its prefix. Inputs and targets are shifted by one position; a causal mask blocks future positions. Many positions can be trained in parallel because the full target sequence is known, even though generation itself is sequential.

## Generate by repeatedly selecting from a distribution

At inference, the model produces next-token logits $z$. Temperature sampling uses $\operatorname{softmax}(z/T)$: lower positive $T$ sharpens the distribution, while higher $T$ flattens it. For logits $[2,0]$, temperature $1$ gives probabilities about $[0.881,0.119]$; temperature $2$ gives about $[0.731,0.269]$. Greedy decoding selects the highest-scoring token; sampling draws from a transformed distribution. Top-k or nucleus filtering further limits candidate mass. These settings change output variability, not the model's factual grounding.

Generation stops at a configured condition such as an end token or length limit. The context window bounds the available prefix, and long continuations can accumulate errors or drift from the prompt.

## Evaluate the model and its use separately

Held-out next-token cross-entropy or perplexity evaluates predictive fit on a specified corpus and tokenization. It does not directly measure factual accuracy, reasoning reliability, safety, or usefulness. Application-level evaluation needs representative tasks, appropriate baselines, and checks for contamination or data overlap.

Decoder-only models are convenient for continuation and instruction-style interfaces, while encoder and encoder-decoder models expose different information paths. Architecture choice should follow the task and deployment constraints.

**Takeaway:** GPT-style training predicts the next token under a causal mask; inference feeds generated tokens back as new context. The sequential generation loop and the parallel training objective are related but not identical.